## Test Grid SANOS



In [ ]:
%load_ext autoreload
%autoreload 1

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"], reduced=True).data['SPX']
print(f"/done; columns {data.keys()}")



In [ ]:
%aimport quantdata.arbfree.arbfree

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store, m_o_m
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.arbfree.arbfree import MkArbFree
from dataclasses import dataclass, field
import math
import numpy as np
import time

def fit(data):
    print(data.keys()   )
    num_samples = len(data.dates)

    mkArbFree = MkArbFree("surface", smoothness=0.1, mid_fit_weight=0.1 )

    verbose = Context.all

    ticker = "SPX"
    afs    = {}
    for isample in range(num_samples-1,max(-1,num_samples-200),-20):
        d = data.dates[isample]
        verbose.write(f"{ticker} {d}:")        

        is_valid = data.is_valid[isample,:]
        uttys = np.unique(data.tte_y[isample,:][is_valid])
        r = PrettyObject()
        r.expiries = []
        r.strikes = []
        r.bids = []
        r.asks = []

        for utty in uttys:
            mask = (data.tte_y[isample,:] == utty) & is_valid
            r.expiries.append(utty)
            r.strikes.append(data.pure_strike[isample][mask])
            r.bids.append(data.pure_bid_price[isample][mask])
            r.asks.append(data.pure_ask_price[isample][mask])
            ixs = np.argsort(r.strikes[-1])
            r.strikes[-1] = r.strikes[-1][ixs]
            r.bids[-1] = r.bids[-1][ixs]
            r.asks[-1] = r.asks[-1][ixs]
        
        af = mkArbFree( expiries=r.expiries, strikes=r.strikes, bids=r.bids, asks=r.asks,
                        verbose=verbose(1) )

        afs[d] = af

fit(data)
